In [0]:
def test_running_total(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, sum as _sum

    data = [
        ("C1", "2024-01-01", 100),
        ("C1", "2024-01-02", 200),
        ("C1", "2024-01-03", 300),
        ("C2", "2024-01-01", 400),
        ("C2", "2024-01-02", 150)
    ]

    columns = ["customer_id", "date", "amount"]
    df = spark.createDataFrame(data, columns)

    window_spec = Window.partitionBy("customer_id") \
        .orderBy("date") \
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    result_df = df.withColumn("running_total", _sum(col("amount")).over(window_spec))

    result = {(row["customer_id"], row["date"], row["running_total"]) 
              for row in result_df.collect()}

    expected = {
        ("C1", "2024-01-01", 100),
        ("C1", "2024-01-02", 300),
        ("C1", "2024-01-03", 600),
        ("C2", "2024-01-01", 400),
        ("C2", "2024-01-02", 550)
    }

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

# Call function        
test_running_total(spark)